<a href="https://colab.research.google.com/github/abidhasan5135-maker/Machine_Learning.369/blob/Machine_Branch/Data_Preprocessing_and_Pipeline_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0. Setup and Imports**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


# 📑 Table of Contents

0. Setup and Imports
1. Loading and Exploring the Dataset
2. Train-Test Split
3. Feature Scaling
    - 3.1 Why Scale?
    - 3.2 StandardScaler
    - 3.3 MinMaxScaler (Min-Max Normalization)
4. Encoding Categorical Data
    - 4.1 Label Encoding (For Binary/Two Categories)
    - 4.2 One-Hot Encoding (For Multiple Categories)
5. Train a Simple Model
6. Cross-Validation
7. Complete Pipeline (All Steps Together)
8. Assignment 1: Agriculture Dataset (Regression Practice)
9. Assignment 2: Housing Dataset (Classification Practice)

# **1. Loading and Exploring the Dataset**

In [4]:
df = pd.read_csv('/content/drive/MyDrive/SKILL_MOR_6/diabetes.csv')  # No header in raw CSV

In [5]:
# Basic exploration
print("Dataset Shape:", df.shape)

Dataset Shape: (768, 9)


In [6]:

print("\nFirst 5 rows:")
print(df.head())




First 5 rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [7]:
print("\nDataset Info:")
print(df.info())




Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None


In [8]:
print("\nBasic Statistics:")
print(df.describe())




Basic Statistics:
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    0.476951  
min      0.00000

In [9]:
print("\nCheck for missing values:")
print(df.isnull().sum())




Check for missing values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [10]:
print("\nTarget variable distribution:")
print(df['Outcome'].value_counts())


Target variable distribution:
Outcome
0    500
1    268
Name: count, dtype: int64


# **2. Train-Test Split**

In [11]:
# Separate input (X) and output (y)
X = df.drop('Outcome', axis=1)  # Everything except Outcome
y = df['Outcome']                # Only Outcome

print("X has all patient information (8 columns)")
print("y has diabetes yes/no (1 column)")

# Split: 80% for training, 20% for testing
# stratify=y keeps the same Outcome (yes/no) ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% for test
    stratify=y,       # keep the same Outcome ratio in train and test
    random_state=30 # Same result every time
)

print(f"\nWe will train with: {len(X_train)} patients")
print(f"We will test with: {len(X_test)} patients")



X has all patient information (8 columns)
y has diabetes yes/no (1 column)

We will train with: 614 patients
We will test with: 154 patients


# **3. Feature Scaling**
## **3.1 Why Scale?**

In [12]:
# Problem: Some numbers are big, some are small
print("Before scaling - different ranges:")
print(f"Age: smallest={X_train['Age'].min()}, biggest={X_train['Age'].max()}")
print(f"Insulin: smallest={X_train['Insulin'].min()}, biggest={X_train['Insulin'].max()}")
print("\nAge goes from 21 to 81 (small range)")
print("Insulin goes from 0 to 846 (big range!)")
print("This confuses the model!")

Before scaling - different ranges:
Age: smallest=21, biggest=72
Insulin: smallest=0, biggest=846

Age goes from 21 to 81 (small range)
Insulin goes from 0 to 846 (big range!)
This confuses the model!


## **3.2 StandardScaler**

In [13]:
# Create scaler
scaler = StandardScaler()

# Learn from training data and scale it
X_train_scaled = scaler.fit_transform(X_train)

# Scale test data (just transform, don't fit)
X_test_scaled = scaler.transform(X_test)

print("\nAfter scaling:")
print("All features now have similar range!")
print(f"Example - First patient's age before scaling: {X_train.values[0][7]}")
print(f"Example - First patient's age after scaling: {X_train_scaled[0][7]:.2f}")


After scaling:
All features now have similar range!
Example - First patient's age before scaling: 21.0
Example - First patient's age after scaling: -1.03


## **3.3 MinMaxScaler (Min-Max Normalization)**

In [14]:
from sklearn.preprocessing import MinMaxScaler

# Create MinMaxScaler
minmax_scaler = MinMaxScaler()

# Learn from training data and scale it
X_train_minmax = minmax_scaler.fit_transform(X_train)

# Scale test data
X_test_minmax = minmax_scaler.transform(X_test)

print("\nAfter MinMaxScaler:")
print("All features now between 0 and 1!")
print(f"Example - First patient's age before scaling: {X_train.values[0][7]}")
print(f"Example - First patient's age after scaling: {X_train_minmax[0][7]:.2f}")

# Let's verify MinMaxScaler worked
print("\nChecking MinMaxScaler:")
print(f"  Minimum value: {X_train_minmax.min():.1f} (should be 0)")
print(f"  Maximum value: {X_train_minmax.max():.1f} (should be 1)")


After MinMaxScaler:
All features now between 0 and 1!
Example - First patient's age before scaling: 21.0
Example - First patient's age after scaling: 0.00

Checking MinMaxScaler:
  Minimum value: 0.0 (should be 0)
  Maximum value: 1.0 (should be 1)


# **4. Encoding Categorical Data**

In [15]:
# Let's create a sample dataset with text data
sample_data = pd.DataFrame({
    'age': [25, 30, 35, 40],
    'gender': ['male', 'female', 'male', 'female'],
    'smoker': ['yes', 'no', 'yes', 'no'],
    'disease': [1, 0, 1, 0]
})

print("Original data with text:")
print(sample_data)

Original data with text:
   age  gender smoker  disease
0   25    male    yes        1
1   30  female     no        0
2   35    male    yes        1
3   40  female     no        0


## **4.1 Label Encoding (For Binary/Two Categories)**

In [16]:
from sklearn.preprocessing import LabelEncoder

# For yes/no or male/female (2 categories only)
label_encoder = LabelEncoder()

# Convert 'smoker' column
sample_data['smoker_encoded'] = label_encoder.fit_transform(sample_data['smoker'])
print("\nAfter Label Encoding 'smoker':")
print(sample_data[['smoker', 'smoker_encoded']])
# no = 0, yes = 1

# Convert 'gender' column
sample_data['gender_encoded'] = label_encoder.fit_transform(sample_data['gender'])
print("\nAfter Label Encoding 'gender':")
print(sample_data[['gender', 'gender_encoded']])
# female = 0, male = 1


After Label Encoding 'smoker':
  smoker  smoker_encoded
0    yes               1
1     no               0
2    yes               1
3     no               0

After Label Encoding 'gender':
   gender  gender_encoded
0    male               1
1  female               0
2    male               1
3  female               0


## **4.2 One-Hot Encoding (For Multiple Categories)**

In [17]:
# One-Hot Encoding for 'city' (Indian cities), indexed as 0/1
sample_data['city'] = ['Delhi', 'Mumbai', 'Delhi', 'Kolkata']

city_encoded = pd.get_dummies(sample_data['city'], prefix='city', dtype=int)

sample_data = pd.concat([sample_data, city_encoded], axis=1)
print(sample_data)

   age  gender smoker  disease  smoker_encoded  gender_encoded     city  \
0   25    male    yes        1               1               1    Delhi   
1   30  female     no        0               0               0   Mumbai   
2   35    male    yes        1               1               1    Delhi   
3   40  female     no        0               0               0  Kolkata   

   city_Delhi  city_Kolkata  city_Mumbai  
0           1             0            0  
1           0             0            1  
2           1             0            0  
3           0             1            0  


# **5. Train a Simple Model**

In [18]:
from sklearn.linear_model import LogisticRegression

# Create model
model = LogisticRegression(max_iter=1000)

# Train the model
model.fit(X_train_scaled, y_train)
print("✅ Model trained!")

# Check accuracy on training data
train_score = model.score(X_train_scaled, y_train)
print(f"Training accuracy: {train_score:.1%}")

# Check accuracy on test data
test_score = model.score(X_test_scaled, y_test)
print(f"Test accuracy: {test_score:.1%}")

✅ Model trained!
Training accuracy: 77.4%
Test accuracy: 77.9%


# **6. Cross-Validation**

In [19]:
from sklearn.model_selection import cross_val_score

# Do 5-fold cross-validation
scores = cross_val_score(
    model,           # Our model
    X_train_scaled,  # Training data
    y_train,         # Training labels
    cv=10            # 5 mini-tests
)

print("5 mini-test scores:")
for i in range(10):
    print(f"  Test {i+1}: {scores[i]:.1%}")

print(f"\nAverage: {scores.mean():.1%}")
print(f"This means our model is {scores.mean():.1%} accurate!")

5 mini-test scores:
  Test 1: 66.1%
  Test 2: 79.0%
  Test 3: 71.0%
  Test 4: 67.7%
  Test 5: 75.4%
  Test 6: 77.0%
  Test 7: 75.4%
  Test 8: 83.6%
  Test 9: 83.6%
  Test 10: 85.2%

Average: 76.4%
This means our model is 76.4% accurate!


# **7. Complete Pipeline (All Steps Together)**

In [21]:
# Complete simple pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# 1. Load data
df = pd.read_csv('/content/drive/MyDrive/SKILL_MOR_6/diabetes.csv')
print(f"Loaded {len(df)} patients")

# 2. Split X and y
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# 3. Train-test split (stratify=y keeps the same Outcome ratio in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Training: {len(X_train)}, Testing: {len(X_test)}")
print(f"Outcome ratio - train: {y_train.mean():.1%}, test: {y_test.mean():.1%}")

# 4. Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data scaled")

# 5. Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
print("Model trained")

# 6. Check accuracy
accuracy = model.score(X_test_scaled, y_test)
print(f"Accuracy: {accuracy:.1%}")

Loaded 768 patients
Training: 614, Testing: 154
Outcome ratio - train: 34.9%, test: 35.1%
Data scaled
Model trained
Accuracy: 71.4%


# **8. Assignment 1: Housing Dataset (Classification Practice)**


In [ ]:
# ASSIGNMENT 1 — YOUR CODE HERE
# Follow the numbered steps from the markdown cell above.

# 1. Load and explore the dataset


# 2. Separate X (features) and y (target = 'price')


# 3. Label Encode: mainroad, guestroom, basement, hotwaterheating, prefarea, airconditioning


# 4. One-Hot Encode: furnishingstatus


# 5. Train-test split (80% train, 20% test) — remember stratify=y


# 6. Scale the numeric features with StandardScaler


# 7. Run: train a LogisticRegression model, apply cross validation( 5 fold, 10 fold) and print the accuracy


1. Load and explore the dataset

In [22]:
df = pd.read_csv('/content/drive/MyDrive/SKILL_MOR_6/Housing.csv')

In [23]:
print("Dataset Shape:", df.shape)

Dataset Shape: (545, 13)


In [24]:
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [26]:
df.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [28]:
print("\nCheck for missing values:")
print(df.isnull().sum())


Check for missing values:
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64


In [36]:
df['furnishingstatus'].value_counts()

,count
furnishingstatus,
semi-furnished,227
unfurnished,178
furnished,140


 2. Separate X (features) and y (target = 'price')
